# Topic Modeling of Policy-Cited Literature

This notebook performs topic modeling on the cleaned publication
datasets produced by the data-preparation workflow.

The analysis includes text preprocessing, document-term matrix
construction, LDA model selection, final topic estimation, topic
interpretation, prevalence analysis, and temporal analysis.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import subprocess
import tempfile

# Project directories
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project directory :", PROJECT_DIR)
print("Data directory    :", DATA_DIR)
print("Output directory  :", OUTPUT_DIR)

## 1. Load Cleaned Publication Data

The cleaned Overton and Scopus publication datasets produced by the
data-preparation notebook are loaded separately. The two sources are
retained as distinct datasets rather than merged.

In [ ]:
OVERTON_FILE = (
    DATA_DIR / "overton_full_clean.xlsx"
)

SCOPUS_FILE = (
    DATA_DIR / "scopus_full_clean.xlsx"
)

overton = pd.read_excel(
    OVERTON_FILE
)

scopus = pd.read_excel(
    SCOPUS_FILE
)

print("Overton")
print("-------")
print(f"Documents : {len(overton):,}")
print(f"Columns   : {len(overton.columns):,}")

print("\nScopus")
print("------")
print(f"Documents : {len(scopus):,}")
print(f"Columns   : {len(scopus.columns):,}")

### 1.1 Define the Independent Topic-Modeling Corpora

The Overton and Scopus publication collections are analyzed as
independent corpora. Each corpus therefore receives its own text
preprocessing, document-term matrix, LDA model-selection procedure,
final topic model, and downstream topic analysis.

In [ ]:
corpora = {
    "overton": overton.copy(),
    "scopus": scopus.copy(),
}

for corpus_name, corpus_df in corpora.items():

    print(f"{corpus_name.upper()}")
    print("-" * len(corpus_name))

    print(
        f"Documents          : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Non-missing titles : "
        f"{corpus_df['Title'].notna().sum():,}"
    )

    print(
        f"Non-missing abstracts: "
        f"{corpus_df['Abstract'].notna().sum():,}"
    )

    print(
        f"Missing abstracts  : "
        f"{corpus_df['Abstract'].isna().sum():,}"
    )

    print()

## 2. Prepare Corpora for Topic Modeling

Topic modeling is performed independently for the Overton and Scopus
datasets. Publications without abstracts are excluded because the LDA
models are estimated from abstract text.

In [ ]:
lda_corpora = {}

for corpus_name, corpus_df in corpora.items():

    lda_df = (
        corpus_df[
            corpus_df["Abstract"].notna()
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Remove abstracts that are empty after whitespace stripping.
    lda_df["Abstract"] = (
        lda_df["Abstract"]
        .astype(str)
        .str.strip()
    )

    lda_df = (
        lda_df[
            lda_df["Abstract"] != ""
        ]
        .reset_index(drop=True)
    )

    lda_corpora[corpus_name] = lda_df

    print(corpus_name.upper())
    print("-" * len(corpus_name))
    print(
        f"Input publications : "
        f"{len(corpus_df):,}"
    )
    print(
        f"LDA documents      : "
        f"{len(lda_df):,}"
    )
    print(
        f"Excluded           : "
        f"{len(corpus_df) - len(lda_df):,}"
    )
    print()
    

### 2.1 Corpus Relevance Diagnostic

Before topic modeling, the retrieved publications are screened
diagnostically for terminology associated with electrical power and
energy systems. This step evaluates whether the search results contain
substantial off-domain literature that could distort the latent topic
structure.

In [ ]:
POWER_DOMAIN_TERMS = [
    r"\bpower system",
    r"\bpower systems",
    r"\belectric power",
    r"\belectrical power",
    r"\bpower grid",
    r"\belectric grid",
    r"\belectrical grid",
    r"\bsmart grid",
    r"\bmicrogrid",
    r"\bmicro-grid",
    r"\btransmission system",
    r"\bdistribution system",
    r"\bpower network",
    r"\belectricity",
    r"\bvoltage",
    r"\breactive power",
    r"\bactive power",
    r"\boptimal power flow",
    r"\bload flow",
    r"\bpower flow",
]

power_pattern = re.compile(
    "|".join(POWER_DOMAIN_TERMS),
    flags=re.IGNORECASE,
)

for corpus_name, corpus_df in lda_corpora.items():

    relevant_mask = (
        corpus_df["Title"]
        .fillna("")
        .str.contains(
            power_pattern,
            regex=True,
        )
        |
        corpus_df["Abstract"]
        .fillna("")
        .str.contains(
            power_pattern,
            regex=True,
        )
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Total documents             : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Power-domain terminology    : "
        f"{relevant_mask.sum():,}"
    )

    print(
        f"No power-domain terminology : "
        f"{(~relevant_mask).sum():,}"
    )

    print(
        f"Share with domain terminology: "
        f"{relevant_mask.mean() * 100:.2f}%"
    )

    print()

In [ ]:
relevance_diagnostics = {}

for corpus_name, corpus_df in lda_corpora.items():

    text = (
        corpus_df["Title"].fillna("")
        + " "
        + corpus_df["Abstract"].fillna("")
    )

    relevant_mask = text.str.contains(
        power_pattern,
        regex=True,
    )

    diagnostic_df = (
        corpus_df.loc[
            ~relevant_mask,
            [
                "Title",
                "Year",
                "DOI",
                "Abstract",
            ],
        ]
        .copy()
    )

    relevance_diagnostics[
        corpus_name
    ] = diagnostic_df

    print(f"\n{corpus_name.upper()}")
    print("-" * len(corpus_name))

    display(
        diagnostic_df[
            [
                "Title",
                "Year",
                "DOI",
            ]
        ]
        .sample(
            n=min(
                30,
                len(diagnostic_df),
            ),
            random_state=123,
        )
        .reset_index(drop=True)
    )

### 2.2 Domain-Relevance Screening

To support a meaningful comparison between policy-facing and academic
literature, both corpora are restricted to publications relevant to
the power and energy systems domain before topic modeling.

The relevance screen is applied identically to Overton and Scopus.
A broad domain vocabulary is used to capture power-system research
without requiring specific terminology such as "power system" or
"power flow". The screening rule is validated through manual
inspection of retained and excluded records before the filtered
corpora are used for LDA.

In [ ]:
# Broad power-and-energy-system terminology used for relevance screening.
#
# This is intentionally broader than the earlier diagnostic.
# The same rule is applied to both Overton and Scopus.

DOMAIN_TERM_GROUPS = {

    "power_system": [
        r"\bpower systems?\b",
        r"\belectric(?:al)? power\b",
        r"\bpower networks?\b",
        r"\belectric(?:al)? networks?\b",
        r"\bpower grids?\b",
        r"\belectric(?:al)? grids?\b",
        r"\bsmart grids?\b",
        r"\bmicrogrids?\b",
        r"\bmicro-grids?\b",
    ],

    "power_flow_operation": [
    r"\bpower flows?\b",
    r"\bload flows?\b",
    r"\boptimal power flows?\b",
    r"\bopf\b",

    # Dispatch and system operation
    r"\beconomic power dispatch\b",
    r"\beconomic dispatch\b",
    r"\boptimal dispatch\b",
    r"\bpower dispatch\b",
    r"\bunit commitment\b",

    # Loss / operating quantities
    r"\bpower loss(?:es)?\b",
    r"\breal power\b",
    r"\bactive power\b",
    r"\breactive power\b",

    # State / voltage / frequency operation
    r"\bstate estimation\b",
    r"\bvoltage stability\b",
    r"\bvoltage control\b",
    r"\bfrequency control\b",
    r"\bfrequency regulation\b",
    ],

    "transmission_distribution": [
        r"\btransmission systems?\b",
        r"\btransmission networks?\b",
        r"\bdistribution systems?\b",
        r"\bdistribution networks?\b",
        r"\bdistribution grids?\b",
        r"\btransmission grids?\b",
        r"\bdistribution feeders?\b",
        r"\bfeeders?\b",
        r"\bsubstations?\b",
    ],

    "electricity": [
        r"\belectricity\b",
        r"\belectric energy\b",
        r"\belectrical energy\b",
        r"\belectricity markets?\b",
        r"\benergy markets?\b",
        r"\belectric utilities?\b",
        r"\bpower utilities?\b",
    ],

    "generation_resources": [
    r"\bpower generation\b",
    r"\belectricity generation\b",
    r"\bgenerating units?\b",
    r"\bgenerators?\b",
    r"\bdistributed generation\b",
    r"\bdistributed energy resources?\b",
    r"\bder\b",
    r"\bders\b",
    r"\benergy resources?\b",
    ],

    "renewables": [
        r"\brenewable energy\b",
        r"\brenewable generation\b",
        r"\bsolar energy\b",
        r"\bsolar power\b",
        r"\bphotovoltaic\b",
        r"\bphotovoltaics\b",
        r"\bpv systems?\b",
        r"\bwind energy\b",
        r"\bwind power\b",
        r"\bwind farms?\b",
        r"\bwind turbines?\b",
    ],

    "storage_ev": [
        r"\benergy storage\b",
        r"\bbattery storage\b",
        r"\bbattery energy storage\b",
        r"\bbess\b",
        r"\belectric vehicles?\b",
        r"\bev charging\b",
        r"\bvehicle-to-grid\b",
        r"\bv2g\b",
    ],

    "power_electronics": [
        r"\bpower electronics\b",
        r"\binverters?\b",
        r"\bconverters?\b",
        r"\bac[- ]dc\b",
        r"\bdc[- ]ac\b",
    ],

    "load_demand": [
        r"\belectric(?:al)? loads?\b",
        r"\bload demand\b",
        r"\belectricity demand\b",
        r"\benergy demand\b",
        r"\bload forecasting\b",
        r"\bdemand response\b",
        r"\bdemand-side management\b",
    ],

    "energy_system": [
    r"\benergy systems?\b",
    r"\bintegrated energy systems?\b",
    r"\bmulti-energy systems?\b",
    r"\bmultienergy systems?\b",
    r"\benergy management systems?\b",
    r"\benergy management\b",
    ],

    "market_reliability": [
    r"\blocational marginal pric(?:e|es|ing)\b",
    r"\blmp\b",
    r"\belectricity pric(?:e|es|ing)\b",
    r"\benergy pric(?:e|es|ing)\b",
    r"\bpower system reliability\b",
    r"\bgrid reliability\b",
    r"\bline failures?\b",
    r"\btransmission line failures?\b",
    r"\bpower system security\b",
    r"\bgrid security\b",
    ],
}

In [ ]:
domain_patterns = {
    group: re.compile(
        "|".join(patterns),
        flags=re.IGNORECASE,
    )
    for group, patterns in DOMAIN_TERM_GROUPS.items()
}

domain_screening = {}

for corpus_name, corpus_df in lda_corpora.items():

    screening_df = corpus_df.copy()

    screening_text = (
        screening_df["Title"]
        .fillna("")
        .astype(str)
        + " "
        + screening_df["Abstract"]
        .fillna("")
        .astype(str)
    )

    matched_columns = []

    for group, pattern in domain_patterns.items():

        column = f"match_{group}"

        screening_df[column] = (
            screening_text.str.contains(
                pattern,
                regex=True,
            )
        )

        matched_columns.append(column)

    # Number of different domain concept groups matched.
    screening_df[
        "Domain_Group_Count"
    ] = (
        screening_df[
            matched_columns
        ]
        .sum(axis=1)
    )

    # Initial broad relevance rule:
    # at least one substantive power/energy-domain group.
    screening_df[
        "Domain_Relevant"
    ] = (
        screening_df[
            "Domain_Group_Count"
        ] >= 1
    )

    domain_screening[
        corpus_name
    ] = screening_df

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Total documents       : "
        f"{len(screening_df):,}"
    )

    print(
        f"Domain relevant       : "
        f"{screening_df['Domain_Relevant'].sum():,}"
    )

    print(
        f"Not domain relevant   : "
        f"{(~screening_df['Domain_Relevant']).sum():,}"
    )

    print(
        f"Retention rate        : "
        f"{screening_df['Domain_Relevant'].mean() * 100:.2f}%"
    )

    print()

In [ ]:
domain_group_summary = []

for corpus_name, screening_df in domain_screening.items():

    for group in DOMAIN_TERM_GROUPS:

        count = int(
            screening_df[
                f"match_{group}"
            ].sum()
        )

        domain_group_summary.append({
            "Corpus": corpus_name.capitalize(),
            "Domain_Group": group,
            "Documents": count,
            "Percent": (
                count
                / len(screening_df)
                * 100
            ),
        })

domain_group_summary = pd.DataFrame(
    domain_group_summary
)

domain_group_summary.pivot(
    index="Domain_Group",
    columns="Corpus",
    values="Documents",
)

In [ ]:
for corpus_name, screening_df in domain_screening.items():

    print(f"\n{'=' * 70}")
    print(corpus_name.upper())
    print(f"{'=' * 70}")

    retained = screening_df[
        screening_df["Domain_Relevant"]
    ]

    rejected = screening_df[
        ~screening_df["Domain_Relevant"]
    ]

    print("\nRANDOM RETAINED DOCUMENTS")
    print("-------------------------")

    display(
        retained[
            [
                "Title",
                "Year",
                "DOI",
                "Domain_Group_Count",
            ]
        ]
        .sample(
            n=min(20, len(retained)),
            random_state=123,
        )
        .reset_index(drop=True)
    )

    print("\nRANDOM REJECTED DOCUMENTS")
    print("-------------------------")

    display(
        rejected[
            [
                "Title",
                "Year",
                "DOI",
            ]
        ]
        .sample(
            n=min(20, len(rejected)),
            random_state=123,
        )
        .reset_index(drop=True)
    )

### 2.3 Final Domain-Filtered Corpora

In [ ]:
lda_corpora_filtered = {}

for corpus_name, screening_df in domain_screening.items():

    filtered_df = (
        screening_df[
            screening_df["Domain_Relevant"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    lda_corpora_filtered[
        corpus_name
    ] = filtered_df

    original_n = len(
        lda_corpora[corpus_name]
    )

    filtered_n = len(
        filtered_df
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Original documents : "
        f"{original_n:,}"
    )

    print(
        f"Retained documents : "
        f"{filtered_n:,}"
    )

    print(
        f"Excluded documents : "
        f"{original_n - filtered_n:,}"
    )

    print(
        f"Retention rate     : "
        f"{filtered_n / original_n * 100:.2f}%"
    )

    print()

## 3. R Text-Processing Backend

The abstract corpora are preprocessed using R `tm` and `SnowballC`
through `Rscript`. This preserves the text-processing methodology used
for the LDA analysis while allowing the complete workflow to be
controlled from Python.

In [ ]:
from pathlib import Path
import subprocess

RSCRIPT = Path(
    r"/nfs/mfirdausi/miniconda3/envs/pytorch/bin/Rscript"
)

if not RSCRIPT.exists():
    raise FileNotFoundError(
        f"Rscript not found: {RSCRIPT}"
    )

# Check required R packages.
r_package_check = subprocess.run(
    [
        str(RSCRIPT),
        "-e",
        (
            'pkgs <- c("tm", "SnowballC", "slam", "topicmodels"); '
            'ok <- sapply(pkgs, requireNamespace, quietly=TRUE); '
            'cat(paste(pkgs, ok, sep="="), sep="\\n")'
        ),
    ],
    capture_output=True,
    text=True,
    check=True,
)

print("Rscript:")
print(RSCRIPT)

print("\nRequired R packages:")
print(r_package_check.stdout)

### 3.1 Text-Preprocessing Configuration

The Overton and Scopus corpora are processed using an identical text
preprocessing configuration to support direct comparison between the
two independently estimated topic models.

Standard English stopwords are supplemented with terms appearing
explicitly in the literature-search query because these terms define
the corpus but provide limited information for distinguishing latent
topics.

In [ ]:
# Search-query terms removed from both corpora.
# change this based on your topic

QUERY_STOPWORDS = [
    # Search-query terms
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation",

    # Publisher/copyright boilerplate
    "©",
]

# Initial DTM configuration.
MIN_TERM_LENGTH = 3
MIN_DOC_FREQ = 3

print("Query-specific stopwords:")
for word in QUERY_STOPWORDS:
    print(f"  - {word}")

print("\nDTM configuration")
print("-----------------")
print("Minimum term length     :", MIN_TERM_LENGTH)
print("Minimum document freq.  :", MIN_DOC_FREQ)

### 3.2 Preprocess Abstracts with R `tm`

The same R `tm` and `SnowballC` preprocessing pipeline is applied
independently to the Overton and Scopus abstracts. Processing includes
lowercasing, punctuation and number removal, whitespace normalization,
English and query-specific stopword removal, and English Snowball
stemming.

The resulting corpora are used to inspect vocabulary characteristics
before the final document-term matrices are constructed.

In [ ]:
R_PREPROCESS_DIR = OUTPUT_DIR / "r_preprocessing"

R_PREPROCESS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

R_PREPROCESS_SCRIPT = (
    R_PREPROCESS_DIR / "preprocess_corpus.R"
)

r_preprocess_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file  <- args[1]
output_file <- args[2]

suppressPackageStartupMessages({
    library(tm)
    library(SnowballC)
})

# ------------------------------------------------------------
# Load abstracts
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

abstracts <- data$Abstract

# ------------------------------------------------------------
# Shared stopwords
# ------------------------------------------------------------

custom_stops <- c(
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation",
    "©"
)

all_stops <- unique(
    c(
        tm::stopwords("english"),
        custom_stops
    )
)

# ------------------------------------------------------------
# tm preprocessing
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(abstracts)
)

corpus <- tm_map(
    corpus,
    content_transformer(tolower)
)

corpus <- tm_map(
    corpus,
    removePunctuation
)

corpus <- tm_map(
    corpus,
    removeNumbers
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

corpus <- tm_map(
    corpus,
    removeWords,
    all_stops
)

# Remove copyright symbol explicitly.
corpus <- tm_map(
    corpus,
    content_transformer(
        function(x) gsub(
            "©",
            " ",
            x,
            fixed = TRUE
        )
    )
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

corpus <- tm_map(
    corpus,
    stemDocument,
    language = "english"
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

processed <- vapply(
    corpus,
    as.character,
    character(1)
)

result <- data.frame(
    document_id = seq_along(processed),
    processed_text = processed,
    stringsAsFactors = FALSE
)

write.csv(
    result,
    output_file,
    row.names = FALSE,
    fileEncoding = "UTF-8"
)
'''

R_PREPROCESS_SCRIPT.write_text(
    r_preprocess_code,
    encoding="utf-8",
)

print(
    "Created R preprocessing script:",
    R_PREPROCESS_SCRIPT
)

In [ ]:
processed_corpora = {}

for corpus_name, corpus_df in lda_corpora_filtered.items():

    input_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_abstracts.csv"
    )

    output_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_processed.csv"
    )
    # ---------------------------------------------------------
    # Use cached processed corpus if it already exists
    # ---------------------------------------------------------

    if output_file.exists():

        print(
            f"Loading cached {corpus_name.upper()} "
            f"processed corpus ..."
        )

    else:

        print(
            f"Processing {corpus_name.upper()} with R ..."
        )

        # Export abstracts only when preprocessing is required.
        corpus_df[
            ["Abstract"]
        ].to_csv(
            input_file,
            index=False,
            encoding="utf-8",
        )

        run = subprocess.run(
            [
                str(RSCRIPT),
                str(R_PREPROCESS_SCRIPT),
                str(input_file),
                str(output_file),
            ],
            capture_output=True,
            text=True,
            check=True,
        )

    # ---------------------------------------------------------
    # Load processed corpus
    # ---------------------------------------------------------

    processed_df = pd.read_csv(
        output_file,
        keep_default_na=False,
    )

    processed_corpora[
        corpus_name
    ] = processed_df

    print(
        f"Documents processed : "
        f"{len(processed_df):,}"
    )

    print(
        f"Empty documents     : "
        f"{(processed_df['processed_text'].str.strip() == '').sum():,}"
    )

    print()

In [ ]:
for corpus_name, processed_df in processed_corpora.items():

    token_counts = (
        processed_df[
            "processed_text"
        ]
        .str.split()
        .str.len()
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents       : "
        f"{len(processed_df):,}"
    )

    print(
        f"Total tokens    : "
        f"{int(token_counts.sum()):,}"
    )

    print(
        f"Minimum tokens  : "
        f"{int(token_counts.min()):,}"
    )

    print(
        f"Median tokens   : "
        f"{token_counts.median():.0f}"
    )

    print(
        f"Mean tokens     : "
        f"{token_counts.mean():.1f}"
    )

    print(
        f"Maximum tokens  : "
        f"{int(token_counts.max()):,}"
    )

    print()

### 3.3 Inspect Frequent Terms

The most frequent terms remaining after preprocessing are inspected
before constructing the final document-term matrices. This diagnostic
is used to identify high-frequency generic terms that provide little
thematic discrimination and may therefore warrant inclusion in the
shared custom stopword list.

In [ ]:
from collections import Counter

term_frequency_tables = {}

for corpus_name, processed_df in processed_corpora.items():

    term_frequency = Counter(
        token
        for text in processed_df["processed_text"]
        for token in text.split()
    )

    top_terms = pd.DataFrame(
        term_frequency.most_common(50),
        columns=[
            "term",
            "frequency",
        ],
    )

    term_frequency_tables[
        corpus_name
    ] = top_terms

    print(f"\n{corpus_name.upper()}")
    print("-" * len(corpus_name))

    display(
        top_terms.head(30)
    )

## 4. Document-Term Matrix Construction

Separate document-term matrices are constructed for the Overton and
Scopus corpora using R `tm`. The same preprocessing and vocabulary
filtering rules are applied to both corpora.

Terms shorter than three characters and terms occurring in fewer than
three documents are excluded. The resulting matrix dimensions and
sparsity are inspected before LDA model selection.

In [ ]:
R_DTM_SCRIPT = (
    R_PREPROCESS_DIR / "construct_dtm.R"
)

r_dtm_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file   <- args[1]
summary_file <- args[2]
terms_file   <- args[3]

suppressPackageStartupMessages({
    library(tm)
    library(slam)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- 3

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

# Remove empty documents before DTM construction.
valid <- !is.na(processed) & nzchar(trimws(processed))

processed <- processed[valid]

# ------------------------------------------------------------
# Construct tm corpus
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Initial DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Minimum document frequency
# ------------------------------------------------------------

doc_freq <- slam::col_sums(
    dtm > 0
)

keep <- doc_freq >= MIN_DOC_FREQ

dtm <- dtm[
    ,
    keep
]

doc_freq <- doc_freq[
    keep
]

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_docs <- dtm$nrow
n_terms <- dtm$ncol
n_nonzero <- length(dtm$v)
n_tokens <- sum(dtm$v)

density <- (
    n_nonzero
    / (n_docs * n_terms)
    * 100
)

summary_result <- data.frame(
    Documents = n_docs,
    Vocabulary = n_terms,
    Nonzero_entries = n_nonzero,
    Total_tokens = n_tokens,
    Density_percent = density
)

write.csv(
    summary_result,
    summary_file,
    row.names = FALSE
)

term_result <- data.frame(
    Term = Terms(dtm),
    Document_frequency = as.numeric(doc_freq),
    stringsAsFactors = FALSE
)

write.csv(
    term_result,
    terms_file,
    row.names = FALSE
)
'''

R_DTM_SCRIPT.write_text(
    r_dtm_code,
    encoding="utf-8",
)

print(
    "Created R DTM script:",
    R_DTM_SCRIPT
)

In [ ]:
dtm_summaries = {}
dtm_term_tables = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    # Domain-filtered processed corpus
    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_processed.csv"
    )

    # Keep domain-filtered DTM diagnostics separate
    # from the previous unfiltered analysis.
    summary_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_dtm_summary.csv"
    )

    terms_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_dtm_terms.csv"
    )

    subprocess.run(
        [
            str(RSCRIPT),
            str(R_DTM_SCRIPT),
            str(processed_file),
            str(summary_file),
            str(terms_file),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    summary_df = pd.read_csv(
        summary_file
    )

    terms_df = pd.read_csv(
        terms_file
    )

    dtm_summaries[
        corpus_name
    ] = summary_df

    dtm_term_tables[
        corpus_name
    ] = terms_df

    row = summary_df.iloc[0]

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents       : "
        f"{int(row['Documents']):,}"
    )

    print(
        f"Vocabulary      : "
        f"{int(row['Vocabulary']):,}"
    )

    print(
        f"Nonzero entries : "
        f"{int(row['Nonzero_entries']):,}"
    )

    print(
        f"Total tokens    : "
        f"{int(row['Total_tokens']):,}"
    )

    print(
        f"Matrix density  : "
        f"{row['Density_percent']:.4f}%"
    )

    print()

### 4.1 Vocabulary Filtering

In [ ]:
# Compare vocabulary sizes under alternative document-frequency
# thresholds using the document frequencies already calculated in R.

DF_THRESHOLDS = [
    3,
    5,
    10,
    15,
    20,
    25,
    50,
    100,
]

df_threshold_results = []

for corpus_name, terms_df in dtm_term_tables.items():

    for threshold in DF_THRESHOLDS:

        retained = (
            terms_df["Document_frequency"]
            >= threshold
        ).sum()

        df_threshold_results.append({
            "Corpus": corpus_name.capitalize(),
            "Min_Document_Frequency": threshold,
            "Retained_Terms": int(retained),
        })

df_threshold_results = pd.DataFrame(
    df_threshold_results
)

df_threshold_pivot = (
    df_threshold_results
    .pivot(
        index="Min_Document_Frequency",
        columns="Corpus",
        values="Retained_Terms",
    )
)

df_threshold_pivot

In [ ]:
for corpus_name, terms_df in dtm_term_tables.items():

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    n_docs = int(
        dtm_summaries[
            corpus_name
        ].iloc[0]["Documents"]
    )

    for threshold in DF_THRESHOLDS:

        retained = (
            terms_df["Document_frequency"]
            >= threshold
        ).sum()

        print(
            f"DF >= {threshold:3d} "
            f"({threshold / n_docs * 100:5.3f}% docs)"
            f" : {retained:6,d} terms"
        )

    print()

Based on the threshold analysis, the final vocabulary filter is defined
proportionally rather than using a common absolute document-frequency
count. Terms must occur in at least 0.2% of documents within each
corpus.

This corresponds to a minimum document frequency of 11 documents for
Overton and 25 documents for Scopus. The proportional rule provides
comparable vocabulary filtering despite the different corpus sizes.

### 4.2 Construct the Final Document-Term Matrices

The final document-term matrices are constructed using a minimum term
length of three characters and a minimum document frequency equal to
0.2% of the documents in each corpus.

In [ ]:
R_DTM_SCRIPT = (
    R_PREPROCESS_DIR / "construct_dtm.R"
)

r_dtm_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file   <- args[1]
summary_file <- args[2]
terms_file   <- args[3]
min_doc_freq <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

# Remove empty documents before DTM construction.
valid <- !is.na(processed) & nzchar(trimws(processed))

processed <- processed[valid]

# ------------------------------------------------------------
# Construct tm corpus
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Initial DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Minimum document frequency
# ------------------------------------------------------------

doc_freq <- slam::col_sums(
    dtm > 0
)

keep <- doc_freq >= MIN_DOC_FREQ

dtm <- dtm[
    ,
    keep
]

doc_freq <- doc_freq[
    keep
]

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_docs <- dtm$nrow
n_terms <- dtm$ncol
n_nonzero <- length(dtm$v)
n_tokens <- sum(dtm$v)

density <- (
    n_nonzero
    / (n_docs * n_terms)
    * 100
)

summary_result <- data.frame(
    Documents = n_docs,
    Vocabulary = n_terms,
    Nonzero_entries = n_nonzero,
    Total_tokens = n_tokens,
    Density_percent = density
)

write.csv(
    summary_result,
    summary_file,
    row.names = FALSE
)

term_result <- data.frame(
    Term = Terms(dtm),
    Document_frequency = as.numeric(doc_freq),
    stringsAsFactors = FALSE
)

write.csv(
    term_result,
    terms_file,
    row.names = FALSE
)
'''

R_DTM_SCRIPT.write_text(
    r_dtm_code,
    encoding="utf-8",
)

print(
    "Created R DTM script:",
    R_DTM_SCRIPT
)


In [ ]:
import math

MIN_DOC_PERCENT = 0.002

final_dtm_summaries = {}
final_dtm_term_tables = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_processed.csv"
    )

    n_documents = len(
        processed_corpora[corpus_name]
    )

    min_doc_freq = math.ceil(
        n_documents * MIN_DOC_PERCENT
    )

    summary_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_final_dtm_summary.csv"
    )

    terms_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_final_dtm_terms.csv"
    )

    subprocess.run(
        [
            str(RSCRIPT),
            str(R_DTM_SCRIPT),
            str(processed_file),
            str(summary_file),
            str(terms_file),
            str(min_doc_freq),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    summary_df = pd.read_csv(
        summary_file
    )

    terms_df = pd.read_csv(
        terms_file
    )

    final_dtm_summaries[
        corpus_name
    ] = summary_df

    final_dtm_term_tables[
        corpus_name
    ] = terms_df

    row = summary_df.iloc[0]

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents            : "
        f"{int(row['Documents']):,}"
    )

    print(
        f"Minimum document freq: "
        f"{min_doc_freq:,}"
    )

    print(
        f"DF threshold         : "
        f"{min_doc_freq / n_documents * 100:.3f}%"
    )

    print(
        f"Vocabulary           : "
        f"{int(row['Vocabulary']):,}"
    )

    print(
        f"Nonzero entries      : "
        f"{int(row['Nonzero_entries']):,}"
    )

    print(
        f"Total tokens         : "
        f"{int(row['Total_tokens']):,}"
    )

    print(
        f"Matrix density       : "
        f"{row['Density_percent']:.4f}%"
    )

    print()

## 5. LDA Topic Modeling and Model Selection

LDA models are estimated independently for the domain-filtered Overton
and Scopus corpora using R `topicmodels` with Gibbs sampling.

For model selection, each corpus is divided reproducibly into 80%
training and 20% held-out documents using a fixed random seed.
Candidate topic numbers are evaluated using held-out predictive
performance together with topic coherence and distinctiveness.

In [ ]:
LDA_DIR = OUTPUT_DIR / "lda"

LDA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("LDA output directory:")
print(LDA_DIR)

### 5.1 Training and Held-Out Splits

An independent 80/20 split is generated for each corpus using R's
random-number generator with seed 123. The same sampling procedure is
therefore applied to both datasets.

In [ ]:
RANDOM_SEED = 123

lda_splits = {}

for corpus_name, summary_df in final_dtm_summaries.items():

    n_documents = int(
        summary_df.iloc[0]["Documents"]
    )

    split_result = subprocess.run(
        [
            str(RSCRIPT),
            "-e",
            (
                f"set.seed({RANDOM_SEED}); "
                f"n <- {n_documents}; "
                "train_id <- sample("
                "seq_len(n), "
                "size=floor(0.8*n), "
                "replace=FALSE"
                "); "
                'cat(train_id, sep=",")'
            ),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    # Keep R's 1-based indices because these will
    # subsequently be passed back to R topicmodels.
    train_id = np.fromstring(
        split_result.stdout.strip(),
        sep=",",
        dtype=int,
    )

    all_id = np.arange(
        1,
        n_documents + 1
    )

    test_id = np.setdiff1d(
        all_id,
        train_id
    )

    lda_splits[corpus_name] = {
        "train_id": train_id,
        "test_id": test_id,
    }

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Total documents    : "
        f"{n_documents:,}"
    )

    print(
        f"Training documents : "
        f"{len(train_id):,}"
    )

    print(
        f"Held-out documents : "
        f"{len(test_id):,}"
    )

    print(
        f"Overlap             : "
        f"{len(np.intersect1d(train_id, test_id))}"
    )

    print()

### 5.2 Coarse Topic-Number Search

A coarse search is performed to identify a plausible range for the
number of latent topics in each domain-filtered corpus. Short Gibbs
chains are used at this screening stage to limit computational cost.

The final corpus-specific vocabulary thresholds established in Section
4 are retained throughout model selection.

In [ ]:
K_COARSE = [
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

COARSE_BURN_IN = 50
COARSE_ITERATIONS = 100
COARSE_THIN = 10

MIN_DOC_PERCENT = 0.002

print("Coarse K values :", K_COARSE)
print("Burn-in         :", COARSE_BURN_IN)
print("Iterations      :", COARSE_ITERATIONS)
print("Thin            :", COARSE_THIN)

In [ ]:
R_COARSE_SEARCH_SCRIPT = (
    LDA_DIR / "domain_filtered_coarse_k_search.R"
)

r_coarse_search_code = r'''
args <- commandArgs(trailingOnly = TRUE)

processed_file <- args[1]
train_file     <- args[2]
output_file    <- args[3]
min_doc_freq   <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

K_VALUES <- c(
    5, 10, 15, 20,
    25, 30, 40, 50
)

BURN_IN <- 50
ITERATIONS <- 100
THIN <- 10

# ------------------------------------------------------------
# Load domain-filtered processed corpus
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Reconstruct FINAL DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

doc_freq <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    doc_freq >= MIN_DOC_FREQ
]

# ------------------------------------------------------------
# Train / held-out split
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- setdiff(
    seq_len(dtm$nrow),
    train_id
)

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# Retain terms represented in training.
train_terms <- (
    slam::col_sums(
        dtm_train > 0
    ) > 0
)

dtm_train <- dtm_train[
    ,
    train_terms
]

dtm_test <- dtm_test[
    ,
    train_terms
]

# Remove zero-token documents after training-vocabulary filtering.
train_nonempty <- (
    slam::row_sums(dtm_train) > 0
)

test_nonempty <- (
    slam::row_sums(dtm_test) > 0
)

n_empty_train <- sum(!train_nonempty)
n_empty_test <- sum(!test_nonempty)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum document frequency:",
    MIN_DOC_FREQ,
    "\n"
)

cat(
    "Empty training removed:",
    n_empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    n_empty_test,
    "\n\n"
)

# ------------------------------------------------------------
# Coarse K search
# ------------------------------------------------------------

results <- data.frame(
    K = integer(),
    Perplexity = numeric(),
    Elapsed_seconds = numeric()
)

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "... "
    )

    flush.console()

    start_time <- Sys.time()

    lda_model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = 123,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    heldout_perplexity <- (
        topicmodels::perplexity(
            lda_model,
            newdata = dtm_test
        )
    )

    elapsed <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    results <- rbind(
        results,
        data.frame(
            K = k,
            Perplexity = heldout_perplexity,
            Elapsed_seconds = elapsed
        )
    )

    # Preserve completed K values.
    write.csv(
        results,
        output_file,
        row.names = FALSE
    )

    cat(
        sprintf(
            "perplexity = %.4f | %.2f s\n",
            heldout_perplexity,
            elapsed
        )
    )

    flush.console()
}
'''

R_COARSE_SEARCH_SCRIPT.write_text(
    r_coarse_search_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_COARSE_SEARCH_SCRIPT
)

In [ ]:
import math

corpus_name = "overton"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

train_file = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_train_indices.csv"
)

coarse_output = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_coarse_k_search.csv"
)

# Same proportional vocabulary rule used in Section 4.
n_documents = int(
    final_dtm_summaries[
        corpus_name
    ].iloc[0]["Documents"]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

print("Running domain-filtered Overton coarse K search...")
print("Documents       :", f"{n_documents:,}")
print("Minimum DF      :", min_doc_freq)
print("K values        :", K_COARSE)

coarse_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_COARSE_SEARCH_SCRIPT),
        str(processed_file),
        str(train_file),
        str(coarse_output),
        str(min_doc_freq),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "Return code:",
    coarse_run.returncode
)

if coarse_run.stdout.strip():
    print("\nR output:")
    print(
        coarse_run.stdout
    )

if coarse_run.stderr.strip():
    print("\nR messages:")
    print(
        coarse_run.stderr
    )

# Load completed results.
if coarse_output.exists():

    overton_coarse_results = pd.read_csv(
        coarse_output
    )

    print(
        "\nCompleted K values:",
        overton_coarse_results["K"].tolist()
    )

    display(
        overton_coarse_results
    )

else:

    print(
        "\nNo completed K values were saved."
    )

#### 5.2.1 Extended Overton Search

Because held-out perplexity continued to decrease through \(K=50\),
the screening range is extended to \(K=60,70,80,100\). These models
retain the same short Gibbs-chain settings and are used only to
characterize the model-complexity trend.

In [ ]:
R_COARSE_SEARCH_SCRIPT = (
    LDA_DIR / "overton_domain_filtered_extended_k_search.R"
)

r_coarse_search_code = r'''
args <- commandArgs(trailingOnly = TRUE)

processed_file <- args[1]
train_file     <- args[2]
output_file    <- args[3]
min_doc_freq   <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

K_VALUES <- c(
    60, 70, 80, 100
)

BURN_IN <- 50
ITERATIONS <- 100
THIN <- 10

# ------------------------------------------------------------
# Load domain-filtered processed corpus
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Reconstruct FINAL DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

doc_freq <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    doc_freq >= MIN_DOC_FREQ
]

# ------------------------------------------------------------
# Train / held-out split
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- setdiff(
    seq_len(dtm$nrow),
    train_id
)

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# Retain terms represented in training.
train_terms <- (
    slam::col_sums(
        dtm_train > 0
    ) > 0
)

dtm_train <- dtm_train[
    ,
    train_terms
]

dtm_test <- dtm_test[
    ,
    train_terms
]

# Remove zero-token documents after training-vocabulary filtering.
train_nonempty <- (
    slam::row_sums(dtm_train) > 0
)

test_nonempty <- (
    slam::row_sums(dtm_test) > 0
)

n_empty_train <- sum(!train_nonempty)
n_empty_test <- sum(!test_nonempty)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum document frequency:",
    MIN_DOC_FREQ,
    "\n"
)

cat(
    "Empty training removed:",
    n_empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    n_empty_test,
    "\n\n"
)

# ------------------------------------------------------------
# Coarse K search
# ------------------------------------------------------------

results <- data.frame(
    K = integer(),
    Perplexity = numeric(),
    Elapsed_seconds = numeric()
)

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "... "
    )

    flush.console()

    start_time <- Sys.time()

    lda_model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = 123,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    heldout_perplexity <- (
        topicmodels::perplexity(
            lda_model,
            newdata = dtm_test
        )
    )

    elapsed <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    results <- rbind(
        results,
        data.frame(
            K = k,
            Perplexity = heldout_perplexity,
            Elapsed_seconds = elapsed
        )
    )

    # Preserve completed K values.
    write.csv(
        results,
        output_file,
        row.names = FALSE
    )

    cat(
        sprintf(
            "perplexity = %.4f | %.2f s\n",
            heldout_perplexity,
            elapsed
        )
    )

    flush.console()
}
'''

R_COARSE_SEARCH_SCRIPT.write_text(
    r_coarse_search_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_COARSE_SEARCH_SCRIPT
)

In [ ]:
import math

K_EXTENDED = [
    60,
    70,
    80,
    100,
]

corpus_name = "overton"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

train_file = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_train_indices.csv"
)

# IMPORTANT:
# Save separately from the original K=5,...,50 coarse search.
extended_output = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_extended_k_search.csv"
)

# Same proportional vocabulary rule used in Section 4.
n_documents = int(
    final_dtm_summaries[
        corpus_name
    ].iloc[0]["Documents"]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

print(
    "Running domain-filtered Overton extended K search..."
)

print(
    "Documents       :",
    f"{n_documents:,}"
)

print(
    "Minimum DF      :",
    min_doc_freq
)

print(
    "K values        :",
    K_EXTENDED
)

extended_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_COARSE_SEARCH_SCRIPT),
        str(processed_file),
        str(train_file),
        str(extended_output),
        str(min_doc_freq),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "Return code:",
    extended_run.returncode
)

if extended_run.stdout.strip():

    print("\nR output:")

    print(
        extended_run.stdout
    )

if extended_run.stderr.strip():

    print("\nR messages:")

    print(
        extended_run.stderr
    )

# Load completed extended-search results.
if extended_output.exists():

    overton_extended_results = pd.read_csv(
        extended_output
    )

    print(
        "\nCompleted K values:",
        overton_extended_results[
            "K"
        ].tolist()
    )

    display(
        overton_extended_results
    )

else:

    print(
        "\nNo completed extended K values were saved."
    )

In [ ]:
overton_k_screen = (
    pd.concat(
        [
            overton_coarse_results,
            overton_extended_results,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset="K"
    )
    .sort_values("K")
    .reset_index(drop=True)
)

display(overton_k_screen)

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    overton_k_screen["K"],
    overton_k_screen["Perplexity"],
    marker="o",
)

ax.set_xlabel(
    "Number of Topics (K)"
)

ax.set_ylabel(
    "Held-Out Perplexity"
)

ax.set_title(
    "Overton LDA Topic-Number Screening"
)

ax.grid(
    alpha=0.3
)

plt.tight_layout()
plt.show()

### 5.3 Multi-Criterion Candidate Evaluation

Because held-out perplexity continued to decrease throughout the
screening range, perplexity alone does not provide a finite optimum
for the number of topics.

Candidate models at \(K=30,50,70,\) and \(100\) are therefore
re-estimated using longer Gibbs chains and evaluated using held-out
perplexity, semantic coherence, topic distinctiveness, and thematic
interpretability.

In [ ]:
K_CANDIDATES = [
    30,
    50,
    70,
    100,
]

CANDIDATE_BURN_IN = 500
CANDIDATE_ITERATIONS = 1000
CANDIDATE_THIN = 50

TOP_N_COHERENCE = 10

print("Candidate K values :", K_CANDIDATES)
print("Burn-in           :", CANDIDATE_BURN_IN)
print("Iterations        :", CANDIDATE_ITERATIONS)
print("Thin              :", CANDIDATE_THIN)
print("Coherence top-N   :", TOP_N_COHERENCE)

In [ ]:
R_CANDIDATE_SCRIPT = (
    LDA_DIR
    / "domain_filtered_candidate_evaluation.R"
)

r_candidate_code = r'''
args <- commandArgs(trailingOnly = TRUE)

processed_file <- args[1]
train_file     <- args[2]
output_dir     <- args[3]
min_doc_freq   <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

K_VALUES <- c(
    30, 50, 70, 100
)

BURN_IN <- 500
ITERATIONS <- 1000
THIN <- 50

TOP_N <- 10

dir.create(
    output_dir,
    recursive = TRUE,
    showWarnings = FALSE
)

# ============================================================
# Load domain-filtered processed corpus
# ============================================================

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

corpus <- VCorpus(
    VectorSource(processed)
)

# ============================================================
# Reconstruct final DTM
# ============================================================

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

doc_freq <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    doc_freq >= MIN_DOC_FREQ
]

# ============================================================
# Training / held-out split
# ============================================================

train_id <- read.csv(
    train_file
)$train_id

test_id <- setdiff(
    seq_len(dtm$nrow),
    train_id
)

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# Vocabulary represented in training.
train_terms <- (
    slam::col_sums(
        dtm_train > 0
    ) > 0
)

dtm_train <- dtm_train[
    ,
    train_terms
]

dtm_test <- dtm_test[
    ,
    train_terms
]

# Remove empty documents.
train_nonempty <- (
    slam::row_sums(dtm_train) > 0
)

test_nonempty <- (
    slam::row_sums(dtm_test) > 0
)

n_empty_train <- sum(
    !train_nonempty
)

n_empty_test <- sum(
    !test_nonempty
)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum document frequency:",
    MIN_DOC_FREQ,
    "\n"
)

cat(
    "Empty training removed:",
    n_empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    n_empty_test,
    "\n\n"
)

# ============================================================
# Binary training DTM for semantic coherence
# ============================================================

binary_train <- dtm_train
binary_train$v[] <- 1

term_doc_freq <- slam::col_sums(
    binary_train
)

# ============================================================
# Semantic coherence
# ============================================================

topic_coherence <- function(
    top_indices,
    binary_dtm,
    term_df
) {

    score <- 0

    for (m in 2:length(top_indices)) {

        for (l in 1:(m - 1)) {

            term_m <- top_indices[m]
            term_l <- top_indices[l]

            docs_m <- binary_dtm$i[
                binary_dtm$j == term_m
            ]

            docs_l <- binary_dtm$i[
                binary_dtm$j == term_l
            ]

            cooccur <- length(
                intersect(
                    docs_m,
                    docs_l
                )
            )

            score <- score + log(
                (cooccur + 1)
                / term_df[term_l]
            )
        }
    }

    score
}

# ============================================================
# Model-level results
# ============================================================

model_results <- data.frame(
    K = integer(),
    Perplexity = numeric(),
    Mean_Coherence = numeric(),
    Median_Coherence = numeric(),
    Min_Coherence = numeric(),
    Mean_Similarity = numeric(),
    Median_Similarity = numeric(),
    Max_Similarity = numeric(),
    Elapsed_seconds = numeric()
)

# ============================================================
# Fit candidate models
# ============================================================

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "...\n"
    )

    flush.console()

    start_time <- Sys.time()

    lda_model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = 123,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    # --------------------------------------------------------
    # Held-out perplexity
    # --------------------------------------------------------

    heldout_perplexity <- (
        topicmodels::perplexity(
            lda_model,
            newdata = dtm_test
        )
    )

    # --------------------------------------------------------
    # Topic-word probabilities
    # --------------------------------------------------------

    posterior_model <- topicmodels::posterior(
        lda_model
    )

    beta <- posterior_model$terms

    # --------------------------------------------------------
    # Coherence and top terms
    # --------------------------------------------------------

    coherence_values <- numeric(k)

    top_term_records <- list()

    for (topic_id in seq_len(k)) {

        top_indices <- order(
            beta[
                topic_id,
            ],
            decreasing = TRUE
        )[seq_len(TOP_N)]

        coherence_values[
            topic_id
        ] <- topic_coherence(
            top_indices,
            binary_train,
            term_doc_freq
        )

        top_term_records[[
            topic_id
        ]] <- data.frame(
            Topic = topic_id,
            Rank = seq_len(TOP_N),
            Term = colnames(beta)[
                top_indices
            ],
            Probability = beta[
                topic_id,
                top_indices
            ]
        )
    }

    top_terms <- do.call(
        rbind,
        top_term_records
    )

    # --------------------------------------------------------
    # Topic cosine similarity
    # --------------------------------------------------------

    beta_norm <- beta / sqrt(
        rowSums(
            beta^2
        )
    )

    similarity_matrix <- (
        beta_norm
        %*%
        t(beta_norm)
    )

    similarity_values <- (
        similarity_matrix[
            upper.tri(
                similarity_matrix
            )
        ]
    )

    # --------------------------------------------------------
    # Runtime
    # --------------------------------------------------------

    elapsed <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    # --------------------------------------------------------
    # Model-level diagnostics
    # --------------------------------------------------------

    model_results <- rbind(
        model_results,
        data.frame(
            K = k,
            Perplexity = heldout_perplexity,
            Mean_Coherence = mean(
                coherence_values
            ),
            Median_Coherence = median(
                coherence_values
            ),
            Min_Coherence = min(
                coherence_values
            ),
            Mean_Similarity = mean(
                similarity_values
            ),
            Median_Similarity = median(
                similarity_values
            ),
            Max_Similarity = max(
                similarity_values
            ),
            Elapsed_seconds = elapsed
        )
    )

    # --------------------------------------------------------
    # Save topic-level coherence
    # --------------------------------------------------------

    write.csv(
        data.frame(
            Topic = seq_len(k),
            Coherence = coherence_values
        ),
        file.path(
            output_dir,
            paste0(
                "coherence_K",
                k,
                ".csv"
            )
        ),
        row.names = FALSE
    )

    # --------------------------------------------------------
    # Save top terms
    # --------------------------------------------------------

    write.csv(
        top_terms,
        file.path(
            output_dir,
            paste0(
                "top_terms_K",
                k,
                ".csv"
            )
        ),
        row.names = FALSE
    )

    # --------------------------------------------------------
    # Save beta
    # --------------------------------------------------------

    beta_df <- data.frame(
        Topic = seq_len(k),
        beta,
        check.names = FALSE
    )

    write.csv(
        beta_df,
        file.path(
            output_dir,
            paste0(
                "beta_K",
                k,
                ".csv"
            )
        ),
        row.names = FALSE
    )

    # Save after every completed candidate.
    write.csv(
        model_results,
        file.path(
            output_dir,
            "candidate_diagnostics.csv"
        ),
        row.names = FALSE
    )

    cat(
        sprintf(
            paste0(
                "K=%d | ",
                "perplexity=%.4f | ",
                "mean coherence=%.4f | ",
                "mean similarity=%.4f | ",
                "%.2f s\n"
            ),
            k,
            heldout_perplexity,
            mean(coherence_values),
            mean(similarity_values),
            elapsed
        )
    )

    flush.console()
}
'''

R_CANDIDATE_SCRIPT.write_text(
    r_candidate_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_CANDIDATE_SCRIPT
)

In [ ]:
corpus_name = "overton"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

train_file = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_train_indices.csv"
)

candidate_output_dir = (
    LDA_DIR
    / "overton_domain_filtered_candidates"
)

candidate_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

n_documents = int(
    final_dtm_summaries[
        corpus_name
    ].iloc[0]["Documents"]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

print(
    "Running domain-filtered Overton candidate evaluation..."
)

print(
    "K values:",
    K_CANDIDATES
)

print(
    "Minimum DF:",
    min_doc_freq
)

candidate_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_CANDIDATE_SCRIPT),
        str(processed_file),
        str(train_file),
        str(candidate_output_dir),
        str(min_doc_freq),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "Return code:",
    candidate_run.returncode
)

if candidate_run.stdout.strip():
    print("\nR output:")
    print(candidate_run.stdout)

if candidate_run.stderr.strip():
    print("\nR messages:")
    print(candidate_run.stderr)

In [ ]:
candidate_diagnostics_file = (
    candidate_output_dir
    / "candidate_diagnostics.csv"
)

print(
    "Diagnostics file:",
    candidate_diagnostics_file
)

if candidate_diagnostics_file.exists():

    overton_candidate_diagnostics = pd.read_csv(
        candidate_diagnostics_file
    )

    display(
        overton_candidate_diagnostics
    )

else:

    print(
        "No completed candidate diagnostics were found."
    )

In [ ]:
overton_model_comparison = (
    overton_candidate_diagnostics[
        [
            "K",
            "Perplexity",
            "Mean_Coherence",
            "Mean_Similarity",
        ]
    ]
    .copy()
)

# Lower perplexity is better.
overton_model_comparison[
    "Perplexity_Rank"
] = (
    overton_model_comparison[
        "Perplexity"
    ]
    .rank(
        ascending=True,
        method="min",
    )
)

# Higher / less-negative coherence is better.
overton_model_comparison[
    "Coherence_Rank"
] = (
    overton_model_comparison[
        "Mean_Coherence"
    ]
    .rank(
        ascending=False,
        method="min",
    )
)

# Lower inter-topic similarity is better.
overton_model_comparison[
    "Similarity_Rank"
] = (
    overton_model_comparison[
        "Mean_Similarity"
    ]
    .rank(
        ascending=True,
        method="min",
    )
)

overton_model_comparison[
    "Mean_Rank"
] = (
    overton_model_comparison[
        [
            "Perplexity_Rank",
            "Coherence_Rank",
            "Similarity_Rank",
        ]
    ]
    .mean(axis=1)
)

overton_model_comparison = (
    overton_model_comparison
    .sort_values(
        [
            "Mean_Rank",
            "K",
        ]
    )
    .reset_index(drop=True)
)

overton_model_comparison

#### 5.3.1 Topic Interpretability and Granularity

The numerical diagnostics reveal a trade-off between predictive
performance and semantic coherence. Candidate models are therefore
inspected for thematic interpretability and excessive topic
fragmentation before selecting the final number of topics.

In [ ]:
overton_candidate_top_terms = {}

for k in K_CANDIDATES:

    top_terms_file = (
        candidate_output_dir
        / f"top_terms_K{k}.csv"
    )

    top_terms_df = pd.read_csv(
        top_terms_file
    )

    overton_candidate_top_terms[k] = (
        top_terms_df
    )

    topic_summary = (
        top_terms_df[
            top_terms_df["Rank"] <= 10
        ]
        .groupby("Topic")["Term"]
        .apply(
            lambda terms: ", ".join(terms)
        )
        .reset_index(
            name="Top_Terms"
        )
    )

    print(f"\nK = {k}")
    print("-" * 20)

    display(topic_summary)

### 5.4 Scopus Topic-Number Selection

The same model-selection procedure is applied independently to the
domain-filtered Scopus corpus. A coarse topic-number search is first
performed using short Gibbs chains and held-out perplexity. The search
is subsequently extended if perplexity continues to decrease at the
upper boundary.

In [ ]:
R_SCOPUS_COARSE_SCRIPT = (
    LDA_DIR
    / "domain_filtered_coarse_k_search.R"
)

print(
    "Scopus coarse-search script:",
    R_SCOPUS_COARSE_SCRIPT
)

In [ ]:
corpus_name = "scopus"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

train_file = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_train_indices.csv"
)

coarse_output = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_coarse_k_search.csv"
)

n_documents = int(
    final_dtm_summaries[
        corpus_name
    ].iloc[0]["Documents"]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

print(
    "Running domain-filtered Scopus coarse K search..."
)

print(
    "Documents       :",
    f"{n_documents:,}"
)

print(
    "Minimum DF      :",
    min_doc_freq
)

print(
    "K values        :",
    K_COARSE
)

scopus_coarse_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_SCOPUS_COARSE_SCRIPT),
        str(processed_file),
        str(train_file),
        str(coarse_output),
        str(min_doc_freq),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "Return code:",
    scopus_coarse_run.returncode
)

if scopus_coarse_run.stdout.strip():

    print("\nR output:")
    print(
        scopus_coarse_run.stdout
    )

if scopus_coarse_run.stderr.strip():

    print("\nR messages:")
    print(
        scopus_coarse_run.stderr
    )

if coarse_output.exists():

    scopus_coarse_results = pd.read_csv(
        coarse_output
    )

    print(
        "\nCompleted K values:",
        scopus_coarse_results[
            "K"
        ].tolist()
    )

    display(
        scopus_coarse_results
    )

else:

    print(
        "\nNo completed Scopus K values were saved."
    )

extended

In [ ]:
R_SCOPUS_EXTENDED_SCRIPT = (
    LDA_DIR
    / "domain_filtered_extended_k_search.R"
)

In [ ]:
K_EXTENDED = [
    60,
    70,
    80,
    100,
]

corpus_name = "scopus"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

train_file = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_train_indices.csv"
)

extended_output = (
    LDA_DIR
    / f"{corpus_name}_domain_filtered_extended_k_search.csv"
)

n_documents = int(
    final_dtm_summaries[
        corpus_name
    ].iloc[0]["Documents"]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

print(
    "Running domain-filtered Scopus extended K search..."
)

print(
    "Documents       :",
    f"{n_documents:,}"
)

print(
    "Minimum DF      :",
    min_doc_freq
)

print(
    "K values        :",
    K_EXTENDED
)

scopus_extended_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_SCOPUS_EXTENDED_SCRIPT),
        str(processed_file),
        str(train_file),
        str(extended_output),
        str(min_doc_freq),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "Return code:",
    scopus_extended_run.returncode
)

if scopus_extended_run.stdout.strip():
    print("\nR output:")
    print(
        scopus_extended_run.stdout
    )

if scopus_extended_run.stderr.strip():
    print("\nR messages:")
    print(
        scopus_extended_run.stderr
    )

if extended_output.exists():

    scopus_extended_results = pd.read_csv(
        extended_output
    )

    print(
        "\nCompleted K values:",
        scopus_extended_results[
            "K"
        ].tolist()
    )

    display(
        scopus_extended_results
    )

else:

    print(
        "\nNo completed extended K values were saved."
    )

#### 5.4.1 Scopus Topic Interpretability and Granularity

The candidate Scopus models are inspected for thematic
interpretability and topic fragmentation. This complements the
quantitative comparison based on held-out perplexity, semantic
coherence, and inter-topic similarity.

In [ ]:
LDA_DIR = OUTPUT_DIR / "lda"

LDA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("LDA directory:", LDA_DIR)

In [ ]:
K_CANDIDATES = [
    30,
    50,
    70,
    100,
]

scopus_candidate_output_dir = (
    LDA_DIR
    / "scopus_domain_filtered_candidates"
)

scopus_candidate_diagnostics = pd.read_csv(
    scopus_candidate_output_dir
    / "candidate_diagnostics.csv"
)

display(
    scopus_candidate_diagnostics
)

In [ ]:
scopus_candidate_top_terms = {}

scopus_candidate_output_dir = (
    LDA_DIR
    / "scopus_domain_filtered_candidates"
)

for k in K_CANDIDATES:

    top_terms_file = (
        scopus_candidate_output_dir
        / f"top_terms_K{k}.csv"
    )

    if not top_terms_file.exists():
        raise FileNotFoundError(
            f"Missing file: {top_terms_file}"
        )

    top_terms_df = pd.read_csv(
        top_terms_file
    )

    scopus_candidate_top_terms[
        k
    ] = top_terms_df

    topic_summary = (
        top_terms_df[
            top_terms_df["Rank"] <= 10
        ]
        .groupby(
            "Topic"
        )["Term"]
        .apply(
            lambda terms: ", ".join(terms)
        )
        .reset_index(
            name="Top_Terms"
        )
    )

    print(f"\nK = {k}")
    print("-" * 20)

    display(
        topic_summary
    )

### 5.5 Final Topic-Number Selection

For both corpora, increasing the number of topics improved held-out
perplexity and reduced mean inter-topic similarity, but progressively
reduced semantic coherence and produced increasingly fine-grained topic
fragmentation.

Inspection of candidate topic structures showed that the 30-topic
models combined several substantively distinct themes, whereas the
70- and 100-topic models increasingly divided established themes into
narrower or partially overlapping components. The 50-topic models
provided a suitable balance between predictive performance, semantic
coherence, topic distinctiveness, and substantive interpretability.

Accordingly, the final number of topics was selected independently as
\(K=50\) for both the Overton and Scopus corpora.

## 6. Final LDA Models

Following model selection, the final LDA models are estimated using all
documents in each domain-filtered corpus. The training/held-out split
used for model selection is no longer required at this stage.

Both final models use 50 topics and are estimated using longer Gibbs
chains to obtain more stable topic-word and document-topic
distributions.

In [2]:
import shutil
from pathlib import Path
import math
import pandas as pd
import subprocess

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

PROJECT_DIR = Path(
    "/nfs/mfirdausi/project/review_paper_2"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "output"
)

R_PREPROCESS_DIR = (
    OUTPUT_DIR
    / "r_preprocessing"
)

LDA_DIR = (
    OUTPUT_DIR
    / "lda"
)

# ---------------------------------------------------------
# R executable
# ---------------------------------------------------------

rscript_path = shutil.which("Rscript")

if rscript_path is None:
    raise FileNotFoundError(
        "Rscript was not found in PATH."
    )

RSCRIPT = Path(
    rscript_path
)

# ---------------------------------------------------------
# Final R script
# ---------------------------------------------------------

R_FINAL_LDA_SCRIPT = (
    LDA_DIR
    / "fit_final_domain_filtered_lda.R"
)

# ---------------------------------------------------------
# Final LDA configuration
# ---------------------------------------------------------

FINAL_K = {
    "overton": 50,
    "scopus": 50,
}

FINAL_BURN_IN = 2000
FINAL_ITERATIONS = 10000
FINAL_THIN = 100
FINAL_RANDOM_SEED = 123

# Final vocabulary rule:
# minimum document frequency = 0.2% of corpus documents.
MIN_DOC_PERCENT = 0.002

# ---------------------------------------------------------
# Validate directories/files
# ---------------------------------------------------------

required_paths = [
    PROJECT_DIR,
    OUTPUT_DIR,
    R_PREPROCESS_DIR,
    LDA_DIR,
    R_FINAL_LDA_SCRIPT,
]

for path in required_paths:

    if not path.exists():
        raise FileNotFoundError(
            f"Required path does not exist: {path}"
        )

# ---------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------

print("Section 6 configuration")
print("-----------------------")

print(
    "Project directory :",
    PROJECT_DIR
)

print(
    "Output directory  :",
    OUTPUT_DIR
)

print(
    "Rscript           :",
    RSCRIPT
)

print(
    "R preprocessing   :",
    R_PREPROCESS_DIR
)

print(
    "LDA directory     :",
    LDA_DIR
)

print(
    "Final R script    :",
    R_FINAL_LDA_SCRIPT
)

print(
    "R script exists   :",
    R_FINAL_LDA_SCRIPT.exists()
)

print()

print(
    "Final K           :",
    FINAL_K
)

print(
    "Burn-in           :",
    FINAL_BURN_IN
)

print(
    "Iterations        :",
    FINAL_ITERATIONS
)

print(
    "Thin              :",
    FINAL_THIN
)

print(
    "Min DF fraction   :",
    MIN_DOC_PERCENT
)

Section 6 configuration
-----------------------
Project directory : /nfs/mfirdausi/project/review_paper_2
Output directory  : /nfs/mfirdausi/project/review_paper_2/output
Rscript           : /nfs/mfirdausi/miniconda3/envs/pytorch/bin/Rscript
R preprocessing   : /nfs/mfirdausi/project/review_paper_2/output/r_preprocessing
LDA directory     : /nfs/mfirdausi/project/review_paper_2/output/lda
Final R script    : /nfs/mfirdausi/project/review_paper_2/output/lda/fit_final_domain_filtered_lda.R
R script exists   : True

Final K           : {'overton': 50, 'scopus': 50}
Burn-in           : 2000
Iterations        : 10000
Thin              : 100
Min DF fraction   : 0.002


### 6.1 Final LDA configuration

In [3]:
FINAL_K = {
    "overton": 50,
    "scopus": 50,
}

FINAL_BURN_IN = 2000
FINAL_ITERATIONS = 10000
FINAL_THIN = 100
FINAL_RANDOM_SEED = 123

print("Final LDA configuration")
print("-----------------------")

for corpus_name, k in FINAL_K.items():
    print(
        f"{corpus_name.upper():8s} : K = {k}"
    )

print()
print("Burn-in    :", FINAL_BURN_IN)
print("Iterations :", FINAL_ITERATIONS)
print("Thin       :", FINAL_THIN)
print("Seed       :", FINAL_RANDOM_SEED)

Final LDA configuration
-----------------------
OVERTON  : K = 50
SCOPUS   : K = 50

Burn-in    : 2000
Iterations : 10000
Thin       : 100
Seed       : 123


### 6.2 Create the final R script

In [4]:
R_FINAL_LDA_SCRIPT = (
    LDA_DIR
    / "fit_final_domain_filtered_lda.R"
)

r_final_lda_code = r'''
args <- commandArgs(trailingOnly = TRUE)

processed_file <- args[1]
output_dir     <- args[2]
min_doc_freq   <- as.integer(args[3])
k              <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

MIN_TERM_LENGTH <- 3

BURN_IN <- 2000
ITERATIONS <- 10000
THIN <- 100
RANDOM_SEED <- 123

TOP_N <- 20

dir.create(
    output_dir,
    recursive = TRUE,
    showWarnings = FALSE
)

# ============================================================
# Load complete processed corpus
# ============================================================

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

original_id <- which(valid)

processed <- processed[
    valid
]

corpus <- VCorpus(
    VectorSource(processed)
)

# ============================================================
# Construct final DTM
# ============================================================

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

doc_freq <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    doc_freq >= min_doc_freq
]

# Remove any documents that become empty after vocabulary filtering.
nonempty <- (
    slam::row_sums(dtm) > 0
)

original_id <- original_id[
    nonempty
]

dtm <- dtm[
    nonempty,
]

cat(
    "Documents:",
    dtm$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm$ncol,
    "\n"
)

cat(
    "Tokens:",
    sum(dtm$v),
    "\n"
)

cat(
    "K:",
    k,
    "\n"
)

cat(
    "Minimum DF:",
    min_doc_freq,
    "\n\n"
)

# ============================================================
# Final Gibbs LDA
# ============================================================

start_time <- Sys.time()

lda_model <- topicmodels::LDA(
    dtm,
    k = k,
    method = "Gibbs",
    control = list(
        seed = RANDOM_SEED,
        burnin = BURN_IN,
        iter = ITERATIONS,
        thin = THIN
    )
)

elapsed <- as.numeric(
    difftime(
        Sys.time(),
        start_time,
        units = "secs"
    )
)

posterior_model <- topicmodels::posterior(
    lda_model
)

beta <- posterior_model$terms
theta <- posterior_model$topics

# ============================================================
# Save beta
# ============================================================

beta_df <- data.frame(
    Topic = seq_len(k),
    beta,
    check.names = FALSE
)

write.csv(
    beta_df,
    file.path(
        output_dir,
        "beta.csv"
    ),
    row.names = FALSE
)

# ============================================================
# Save theta
# ============================================================

theta_df <- data.frame(
    document_id = original_id,
    theta,
    check.names = FALSE
)

write.csv(
    theta_df,
    file.path(
        output_dir,
        "theta.csv"
    ),
    row.names = FALSE
)

# ============================================================
# Top terms
# ============================================================

top_term_records <- list()

for (topic_id in seq_len(k)) {

    ord <- order(
        beta[
            topic_id,
        ],
        decreasing = TRUE
    )[seq_len(TOP_N)]

    top_term_records[[
        topic_id
    ]] <- data.frame(
        Topic = topic_id,
        Rank = seq_len(TOP_N),
        Term = colnames(beta)[
            ord
        ],
        Probability = beta[
            topic_id,
            ord
        ]
    )
}

top_terms <- do.call(
    rbind,
    top_term_records
)

write.csv(
    top_terms,
    file.path(
        output_dir,
        "top_terms.csv"
    ),
    row.names = FALSE
)

# ============================================================
# Dominant topic
# ============================================================

dominant_topic <- max.col(
    theta,
    ties.method = "first"
)

dominant_probability <- apply(
    theta,
    1,
    max
)

document_topics <- data.frame(
    document_id = original_id,
    Dominant_Topic = dominant_topic,
    Dominant_Probability = dominant_probability
)

write.csv(
    document_topics,
    file.path(
        output_dir,
        "document_topics.csv"
    ),
    row.names = FALSE
)

# ============================================================
# Topic prevalence
# ============================================================

topic_prevalence <- colMeans(
    theta
)

prevalence_df <- data.frame(
    Topic = seq_len(k),
    Mean_Probability = topic_prevalence
)

prevalence_df <- prevalence_df[
    order(
        prevalence_df$Mean_Probability,
        decreasing = TRUE
    ),
]

write.csv(
    prevalence_df,
    file.path(
        output_dir,
        "topic_prevalence.csv"
    ),
    row.names = FALSE
)

# ============================================================
# Final summary
# ============================================================

summary_df <- data.frame(
    Documents = dtm$nrow,
    Vocabulary = dtm$ncol,
    Tokens = sum(dtm$v),
    K = k,
    Minimum_DF = min_doc_freq,
    Burn_in = BURN_IN,
    Iterations = ITERATIONS,
    Thin = THIN,
    Seed = RANDOM_SEED,
    Elapsed_seconds = elapsed
)

write.csv(
    summary_df,
    file.path(
        output_dir,
        "model_summary.csv"
    ),
    row.names = FALSE
)

cat(
    "Final LDA completed in",
    round(elapsed, 2),
    "seconds.\n"
)
'''

R_FINAL_LDA_SCRIPT.write_text(
    r_final_lda_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_FINAL_LDA_SCRIPT
)

Created: /nfs/mfirdausi/project/review_paper_2/output/lda/fit_final_domain_filtered_lda.R


### 6.3 Final Overton LDA Model

The final Overton LDA model is estimated on the complete
domain-filtered corpus using \(K=50\). The longer Gibbs chain is used
for final estimation after completion of topic-number selection.

In [5]:
corpus_name = "overton"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

dtm_summary_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_final_dtm_summary.csv"
)

final_output_dir = (
    LDA_DIR
    / "final_overton_k50"
)

final_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# ---------------------------------------------------------
# Load final DTM information from disk
# ---------------------------------------------------------

dtm_summary = pd.read_csv(
    dtm_summary_file
)

n_documents = int(
    dtm_summary.loc[
        0,
        "Documents"
    ]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

k = FINAL_K[
    corpus_name
]

# ---------------------------------------------------------
# Validate required files
# ---------------------------------------------------------

for required_file in [
    processed_file,
    dtm_summary_file,
    R_FINAL_LDA_SCRIPT,
]:

    if not required_file.exists():
        raise FileNotFoundError(
            f"Missing required file: {required_file}"
        )

# ---------------------------------------------------------
# Final Overton LDA
# ---------------------------------------------------------

print("Running final Overton LDA...")
print("-----------------------------")
print("Documents  :", f"{n_documents:,}")
print("K          :", k)
print("Minimum DF :", min_doc_freq)
print("Burn-in    :", FINAL_BURN_IN)
print("Iterations :", FINAL_ITERATIONS)
print("Thin       :", FINAL_THIN)

final_overton_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_FINAL_LDA_SCRIPT),
        str(processed_file),
        str(final_output_dir),
        str(min_doc_freq),
        str(k),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "\nReturn code:",
    final_overton_run.returncode
)

if final_overton_run.stdout.strip():

    print("\nR output:")
    print(
        final_overton_run.stdout
    )

if final_overton_run.stderr.strip():

    print("\nR messages:")
    print(
        final_overton_run.stderr
    )

Running final Overton LDA...
-----------------------------
Documents  : 5,045
K          : 50
Minimum DF : 11
Burn-in    : 2000
Iterations : 10000
Thin       : 100

Return code: 0

R output:
Documents: 5045 
Vocabulary: 2865 
Tokens: 512540 
K: 50 
Minimum DF: 11 

Final LDA completed in 397.06 seconds.



### 6.4 Final Scopus LDA Model

The final Scopus LDA model is estimated on the complete
domain-filtered corpus using \(K=50\) and the same final Gibbs-sampling
configuration used for Overton.

In [ ]:
corpus_name = "scopus"

processed_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_processed.csv"
)

dtm_summary_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_final_dtm_summary.csv"
)

final_output_dir = (
    LDA_DIR
    / "final_scopus_k50"
)

final_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# ---------------------------------------------------------
# Load final DTM information
# ---------------------------------------------------------

dtm_summary = pd.read_csv(
    dtm_summary_file
)

n_documents = int(
    dtm_summary.loc[
        0,
        "Documents"
    ]
)

min_doc_freq = math.ceil(
    n_documents * MIN_DOC_PERCENT
)

k = FINAL_K[
    corpus_name
]

# ---------------------------------------------------------
# Validate required files
# ---------------------------------------------------------

for required_file in [
    processed_file,
    dtm_summary_file,
    R_FINAL_LDA_SCRIPT,
]:

    if not required_file.exists():

        raise FileNotFoundError(
            f"Missing required file: {required_file}"
        )

# ---------------------------------------------------------
# Final Scopus LDA
# ---------------------------------------------------------

print("Running final Scopus LDA...")
print("----------------------------")
print("Documents  :", f"{n_documents:,}")
print("K          :", k)
print("Minimum DF :", min_doc_freq)
print("Burn-in    :", FINAL_BURN_IN)
print("Iterations :", FINAL_ITERATIONS)
print("Thin       :", FINAL_THIN)

final_scopus_run = subprocess.run(
    [
        str(RSCRIPT),
        str(R_FINAL_LDA_SCRIPT),
        str(processed_file),
        str(final_output_dir),
        str(min_doc_freq),
        str(k),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    "\nReturn code:",
    final_scopus_run.returncode
)

if final_scopus_run.stdout.strip():

    print("\nR output:")
    print(
        final_scopus_run.stdout
    )

if final_scopus_run.stderr.strip():

    print("\nR messages:")
    print(
        final_scopus_run.stderr
    )

Running final Scopus LDA...
----------------------------
Documents  : 12,042
K          : 50
Minimum DF : 25
Burn-in    : 2000
Iterations : 10000
Thin       : 100
